# E14 — Supervised SOTA transfer audit

Goal: extend Table~2 of the paper with **six modern supervised anomaly
detectors** under the same regime grid that E9 uses for the six tabular
baselines. The new rows let a reviewer see at a glance whether the
real$\,\to\,$synthetic transfer failure of the simple detectors is rescued
by foundation-model encoders or modern transformer / convolution classifiers.

## Detectors (raw windows of shape `(N, 128, 16)`)
- **MOMENT (frozen)** + binary head — `models/MOMENT-1-large` if available, else HuggingFace.
- **Toto (frozen)** + MLP head — `models/Toto-Open-Base-1.0` if available.
- **Mantis (frozen)** + linear head — `models/Mantis-8M` if available.
- **TimesNet-lite (e2e)** — supervised classification (E8's class with
  `n_classes=2`).
- **InceptionTime-lite (e2e)** — supervised classification (same source).
- **PatchTST (e2e)** — `sota_clones/tslib/models/PatchTST.py` with
  `task_name='classification'` and `num_class=2`.

## Settings (same as E9)
- `controlled_500 / all_origins`
- `controlled_500 / synthetic_only`
- `balanced_detection / synthetic_only`
- `fullscale / synthetic_only`

## Seeds
- 10 seeds; for `controlled_500` the split composition itself varies per seed
  (matches E9); for `balanced_detection` and `fullscale` only the model seed
  changes (validation slice for `fullscale` is re-stratified per seed).

## Output
- `results/E14_sup_per_seed.csv`  — per-seed metrics in the same schema as
  `E9_transfer_per_seed.csv` (so they concatenate trivially).
- `results/E14_sup_summary.csv`   — mean / std across seeds.
- `tables/E14_sup_table.tex`      — wide LaTeX table with the six new rows.


## 0. Bootstrap

In [1]:
# Make _shared importable, set plot defaults, suppress noisy warnings.
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))
from _shared.notebook_helpers import setup_paths, configure_matplotlib
EXPERIMENTS_ROOT = setup_paths()
configure_matplotlib()

import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from _shared.data_utils import (
    load_corpus,
    make_balanced_detection_split,
    make_fullscale_split,
)
from _shared import sota_helpers as sh

PATHS = sh.add_sota_paths()
print('repo_root :', PATHS['repo_root'])

import torch
DEVICE_TORCH = (
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
# Use CPU for Mantis (its internal tensors do not move with .to() reliably).
print('Device     :', DEVICE_TORCH, '| torch', torch.__version__)

HERE = Path('.').resolve()
RESULTS = HERE / 'results'
TABLES  = HERE / 'tables'
EMB_CACHE = HERE / 'embeddings_cache'
RESULTS.mkdir(exist_ok=True); TABLES.mkdir(exist_ok=True)
EMB_CACHE.mkdir(exist_ok=True)
print('Working dir:', HERE)


repo_root : .
Device     : mps | torch 2.10.0
Working dir: experiments/E14_supervised_sota_transfer


## 1. Load corpus and the canonical splits

In [2]:
corpus = load_corpus(verbose=True)
print('corpus N:', corpus.n)
print('X       :', corpus.X.shape, corpus.X.dtype)

SPLITS = {
    'balanced_detection': make_balanced_detection_split(corpus, seed=42),
    'fullscale':          make_fullscale_split(corpus, seed=42),
}
for name, parts in SPLITS.items():
    sizes = {k: int(v.size) for k, v in parts.items()}
    print(f'  {name:>22s} sizes={sizes}')

REGIMES_BY_SETTING = {
    'controlled_500':       ['all_origins', 'synthetic_only'],
    'balanced_detection':   ['synthetic_only'],
    'fullscale':            ['synthetic_only'],
}
SEEDS = list(range(10))


[data_utils] Loading cached corpus from experiments/_shared/cache/telecomts_corpus_v1.pkl
corpus N: 32000
X       : (32000, 128, 16) float32
      balanced_detection sizes={'train': 1580, 'val': 396, 'test': 494}
               fullscale sizes={'train': 25600, 'test': 6400}


## 2. Foundation-model embedding cache

For frozen MOMENT / Toto / Mantis the encoder forward is deterministic, so we
extract embeddings once per (setting, seed) and reuse them across multiple
regimes (the regime only affects which rows feed into the binary head). The
cache key includes the encoder name, setting, and seed.

In [3]:
from typing import Optional


def _emb_path(name: str, setting: str, seed: int, split_part: str) -> Path:
    return EMB_CACHE / f'{name}__{setting}__seed{seed}__{split_part}.npz'


def _load_or_extract(name: str, encoder_fn, X_btc, setting: str, seed: int,
                     split_part: str, **encoder_kwargs):
    path = _emb_path(name, setting, seed, split_part)
    if path.exists():
        d = np.load(path)
        return d['Z'], bool(d['loaded'])
    Z, loaded = encoder_fn(X_btc, **encoder_kwargs)
    np.savez_compressed(path, Z=Z, loaded=np.array(loaded))
    return Z, loaded


def get_embeddings(setting: str, seed: int):
    """Return a dict {encoder_name: (Z_train_pool, Z_val_pool, Z_test, loaded)}.

    The train_pool / val_pool / test indices come from the canonical
    pre-regime split (we slice by regime later, downstream of the cache).
    """
    bundle = sh.make_unsup_dataset(corpus, SPLITS, setting, seed)
    train_idx, val_idx, test_idx = bundle.train_idx, bundle.val_idx, bundle.test_idx
    X_train_btc = corpus.X[train_idx]
    X_val_btc   = corpus.X[val_idx]
    X_test_btc  = corpus.X[test_idx]
    out = {}
    Zm_tr, m_loaded = _load_or_extract('moment', sh.encode_moment, X_train_btc, setting, seed, 'train',
                                        device=DEVICE_TORCH)
    Zm_va, _        = _load_or_extract('moment', sh.encode_moment, X_val_btc,   setting, seed, 'val',
                                        device=DEVICE_TORCH)
    Zm_te, _        = _load_or_extract('moment', sh.encode_moment, X_test_btc,  setting, seed, 'test',
                                        device=DEVICE_TORCH)
    out['MOMENT'] = (Zm_tr, Zm_va, Zm_te, m_loaded)
    Zt_tr, t_loaded = _load_or_extract('toto', sh.encode_toto, X_train_btc, setting, seed, 'train',
                                        device=DEVICE_TORCH)
    Zt_va, _        = _load_or_extract('toto', sh.encode_toto, X_val_btc,   setting, seed, 'val',
                                        device=DEVICE_TORCH)
    Zt_te, _        = _load_or_extract('toto', sh.encode_toto, X_test_btc,  setting, seed, 'test',
                                        device=DEVICE_TORCH)
    out['Toto'] = (Zt_tr, Zt_va, Zt_te, t_loaded)
    Zn_tr, n_loaded = _load_or_extract('mantis', sh.encode_mantis, X_train_btc, setting, seed, 'train',
                                        device='cpu')
    Zn_va, _        = _load_or_extract('mantis', sh.encode_mantis, X_val_btc,   setting, seed, 'val',
                                        device='cpu')
    Zn_te, _        = _load_or_extract('mantis', sh.encode_mantis, X_test_btc,  setting, seed, 'test',
                                        device='cpu')
    out['Mantis'] = (Zn_tr, Zn_va, Zn_te, n_loaded)
    out['__indices__'] = (train_idx, val_idx, test_idx)
    return out


# Eagerly populate the cache so a long e2e run does not get blocked behind
# repeated encoder loads. We extract once per (setting, seed). For
# ``balanced_detection`` and ``fullscale`` the canonical split is fixed at
# seed=42, so foundation embeddings only need to be extracted once for those
# settings (we still vary the per-seed validation slice for fullscale below).
print('Foundation models will lazily populate', EMB_CACHE, 'as needed.')


Foundation models will lazily populate experiments/E14_supervised_sota_transfer/embeddings_cache as needed.


## 3. Per-(setting, regime) train/val/test index builder

Same logic as E9, including the per-seed validation slice for fullscale.

In [4]:
def build_split(setting: str, regime: str, seed: int):
    bundle = sh.make_sup_dataset(corpus, SPLITS, setting, regime, seed)
    return bundle


def regime_filter_indices(parent_idx, parent_y, parent_origin, regime: str):
    """Return the relative positions inside parent_idx that survive the
    regime filter. Used to slice cached embeddings without re-extracting."""
    keep_mask = np.zeros(parent_idx.size, dtype=bool)
    for k, i in enumerate(parent_idx):
        if regime == 'all_origins':
            keep_mask[k] = True
            continue
        if parent_y[k] == 0:
            keep_mask[k] = True
            continue
        origin = parent_origin[k]
        if regime == 'synthetic_only' and origin == 'synthetic':
            keep_mask[k] = True
        elif regime == 'real_only' and origin == 'real':
            keep_mask[k] = True
    return keep_mask


## 4. Main multi-detector loop

For each (setting, regime, seed) we either
1. fit the e2e supervised model directly on the (regime-filtered) raw
   windows, or
2. take the cached frozen embeddings, slice them down by the regime mask,
   and train the binary head.

The result rows match E9's schema so downstream consumers do not need
special-casing for E14.

In [5]:
SUP_DETECTORS = ['MOMENT', 'Toto', 'Mantis', 'TimesNet-lite', 'InceptionTime-lite', 'PatchTST']

per_seed_rows = []
total_runs = sum(len(v) * len(SEEDS) * len(SUP_DETECTORS) for v in REGIMES_BY_SETTING.values())
done = 0
t_global = time.time()
print(f'Total runs to do: {total_runs}')


def _safe_train(name: str, *, X_tr, y_tr, X_va, y_va, X_te, seed: int):
    """E2E supervised models. Returns (val_p, test_p, fit_time, note)."""
    return sh.train_e2e_binary(
        name, X_tr, y_tr, X_va, X_te,
        seed=seed, device=DEVICE_TORCH,
        epochs=10, lr=1e-4, batch_size=64,
        in_channels=corpus.X.shape[2], seq_len=corpus.X.shape[1],
    )


for setting, regimes in REGIMES_BY_SETTING.items():
    for seed in SEEDS:
        cached = None  # lazy: extract foundation embeddings only when needed
        # Pre-build the per-seed bundle once; we will re-filter for each regime.
        parent_bundle = sh.make_unsup_dataset(corpus, SPLITS, setting, seed)
        parent_train_idx = parent_bundle.train_idx
        parent_val_idx   = parent_bundle.val_idx
        parent_test_idx  = parent_bundle.test_idx
        parent_y_train   = corpus.y[parent_train_idx]
        parent_y_val     = corpus.y[parent_val_idx]
        parent_origin_train = corpus.anomaly_origin[parent_train_idx]
        parent_origin_val   = corpus.anomaly_origin[parent_val_idx]
        for regime in regimes:
            keep_train = regime_filter_indices(parent_train_idx, parent_y_train,
                                                parent_origin_train, regime)
            keep_val   = regime_filter_indices(parent_val_idx, parent_y_val,
                                                parent_origin_val, regime)
            train_idx = parent_train_idx[keep_train]
            val_idx   = parent_val_idx[keep_val]
            test_idx  = parent_test_idx
            y_train = corpus.y[train_idx]; y_val = corpus.y[val_idx]; y_test = corpus.y[test_idx]
            test_real_mask  = (corpus.anomaly_origin[test_idx] == 'real')
            test_synth_mask = (corpus.anomaly_origin[test_idx] == 'synthetic')
            test_normal_mask = (corpus.y[test_idx] == 0)

            # Skip if regime removed all of one class -> binary head cannot train.
            if len(np.unique(y_train)) < 2:
                for det in SUP_DETECTORS:
                    per_seed_rows.append(sh.nan_row(
                        setting=setting, regime=regime, detector=det, seed=seed,
                        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
                        test_real_mask=test_real_mask, test_synth_mask=test_synth_mask,
                        note='single_class_train',
                    ))
                    done += 1
                continue

            # ---------- E2E supervised models ---------------------------------
            X_tr_btc = corpus.X[train_idx]
            X_va_btc = corpus.X[val_idx]
            X_te_btc = corpus.X[test_idx]
            for det in ['TimesNet-lite', 'InceptionTime-lite', 'PatchTST']:
                t0 = time.time()
                try:
                    val_p, test_p, fit_dt = _safe_train(
                        det, X_tr=X_tr_btc, y_tr=y_train, X_va=X_va_btc, y_va=y_val,
                        X_te=X_te_btc, seed=seed,
                    )
                    thr = sh.select_threshold(val_p, y_val)
                    row = sh.compute_metrics_row(
                        setting=setting, regime=regime, detector=det, seed=seed,
                        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
                        test_real_mask=test_real_mask, test_synth_mask=test_synth_mask,
                        test_normal_mask=test_normal_mask,
                        threshold=thr, test_scores=test_p, y_test=y_test,
                        fit_time_sec=fit_dt, note='ok_e2e',
                    )
                except Exception as e:
                    print(f'  [{det}] FAILED at setting={setting} regime={regime} seed={seed}: '
                          f'{type(e).__name__}: {e}')
                    row = sh.nan_row(
                        setting=setting, regime=regime, detector=det, seed=seed,
                        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
                        test_real_mask=test_real_mask, test_synth_mask=test_synth_mask,
                        fit_time_sec=time.time()-t0, note=f'error:{type(e).__name__}',
                    )
                per_seed_rows.append(row)
                done += 1
                print(f"  setting={setting:>20s} regime={regime:>16s} seed={seed} "
                      f"det={det:>20s} f1={row['f1']:.3f} auroc={row['auroc']:.3f} "
                      f"real={row['real_det']:.2f} synth={row['synth_det']:.2f} "
                      f"({done}/{total_runs}; elapsed={time.time()-t_global:.0f}s)")

            # ---------- Frozen-encoder + supervised head ---------------------
            if cached is None:
                cached = get_embeddings(setting, seed)
            for det in ['MOMENT', 'Toto', 'Mantis']:
                Zp_tr, Zp_va, Zp_te, loaded = cached[det]
                # Slice cached embeddings down to the regime-filtered rows.
                Z_tr = Zp_tr[keep_train]
                Z_va = Zp_va[keep_val]
                Z_te = Zp_te
                t0 = time.time()
                try:
                    val_p, test_p, fit_dt = sh.train_foundation_binary_head(
                        det, Z_tr, y_train, Z_va, y_val, Z_te,
                        seed=seed, device=DEVICE_TORCH,
                    )
                    thr = sh.select_threshold(val_p, y_val)
                    row = sh.compute_metrics_row(
                        setting=setting, regime=regime, detector=det, seed=seed,
                        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
                        test_real_mask=test_real_mask, test_synth_mask=test_synth_mask,
                        test_normal_mask=test_normal_mask,
                        threshold=thr, test_scores=test_p, y_test=y_test,
                        fit_time_sec=fit_dt,
                        note='ok_foundation' if loaded else 'ok_surrogate',
                    )
                except Exception as e:
                    print(f'  [{det}] FAILED at setting={setting} regime={regime} seed={seed}: '
                          f'{type(e).__name__}: {e}')
                    row = sh.nan_row(
                        setting=setting, regime=regime, detector=det, seed=seed,
                        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
                        test_real_mask=test_real_mask, test_synth_mask=test_synth_mask,
                        fit_time_sec=time.time()-t0, note=f'error:{type(e).__name__}',
                    )
                per_seed_rows.append(row)
                done += 1
                print(f"  setting={setting:>20s} regime={regime:>16s} seed={seed} "
                      f"det={det:>20s} f1={row['f1']:.3f} auroc={row['auroc']:.3f} "
                      f"real={row['real_det']:.2f} synth={row['synth_det']:.2f} "
                      f"({done}/{total_runs}; elapsed={time.time()-t_global:.0f}s)")

        # Checkpoint after each (setting, seed) sweep.
        per_seed_df = pd.DataFrame(per_seed_rows)
        per_seed_df.to_csv(RESULTS / 'E14_sup_per_seed.csv', index=False)

per_seed_df = pd.DataFrame(per_seed_rows)
per_seed_df.to_csv(RESULTS / 'E14_sup_per_seed.csv', index=False)
print('\nFinished. Saved per-seed CSV. shape:', per_seed_df.shape)
per_seed_df.head(8)


Total runs to do: 240
  setting=      controlled_500 regime=     all_origins seed=0 det=       TimesNet-lite f1=0.648 auroc=0.695 real=0.96 synth=0.88 (1/240; elapsed=3s)
  setting=      controlled_500 regime=     all_origins seed=0 det=  InceptionTime-lite f1=0.711 auroc=0.667 real=0.96 synth=0.96 (2/240; elapsed=9s)
  setting=      controlled_500 regime=     all_origins seed=0 det=            PatchTST f1=0.686 auroc=0.653 real=0.92 synth=0.96 (3/240; elapsed=18s)
Loading weights from local directory
Loading weights from local directory
Loading weights from local directory
  setting=      controlled_500 regime=     all_origins seed=0 det=              MOMENT f1=0.667 auroc=0.327 real=1.00 synth=1.00 (4/240; elapsed=47s)
  setting=      controlled_500 regime=     all_origins seed=0 det=                Toto f1=0.796 auroc=0.851 real=0.84 synth=0.64 (5/240; elapsed=50s)
  setting=      controlled_500 regime=     all_origins seed=0 det=              Mantis f1=0.742 auroc=0.879 real=0.88 s

,setting,regime,detector,seed,n_train,n_val,n_test,n_real_test,n_synth_test,threshold,f1,precision,recall,auroc,avg_precision,real_det,synth_det,normal_fpr,fit_time_sec,note
0,controlled_500,all_origins,TimesNet-lite,0,300,100,100,25,25,0.46,0.647887,0.500000,0.92,0.6952,0.743244,0.96,0.88,0.92,1.991967,ok_e2e
1,controlled_500,all_origins,InceptionTime-lite,0,300,100,100,25,25,0.21,0.711111,0.564706,0.96,0.6672,0.681924,0.96,0.96,0.74,5.633492,ok_e2e
2,controlled_500,all_origins,PatchTST,0,300,100,100,25,25,0.17,0.686131,0.540230,0.94,0.6528,0.673490,0.92,0.96,0.80,8.371241,ok_e2e
3,controlled_500,all_origins,MOMENT,0,300,100,100,25,25,0.01,0.666667,0.500000,1.00,0.3272,0.393333,1.00,1.00,1.00,1.938959,ok_foundation
4,controlled_500,all_origins,Toto,0,300,100,100,25,25,0.57,0.795699,0.860465,0.74,0.8512,0.813217,0.84,0.64,0.12,2.648027,ok_foundation
5,controlled_500,all_origins,Mantis,0,300,100,100,25,25,0.63,0.741573,0.846154,0.66,0.8792,0.885932,0.88,0.44,0.12,1.280603,ok_foundation
6,controlled_500,synthetic_only,TimesNet-lite,0,225,75,100,25,25,0.01,0.666667,0.500000,1.00,0.6748,0.716874,1.00,1.00,1.00,1.303416,ok_e2e
7,controlled_500,synthetic_only,InceptionTime-lite,0,225,75,100,25,25,0.50,0.623853,0.576271,0.68,0.6368,0.630852,0.44,0.92,0.50,4.367785,ok_e2e


## 5. Aggregate across seeds

In [6]:
summary = sh.aggregate_per_seed(per_seed_df)
summary.to_csv(RESULTS / 'E14_sup_summary.csv', index=False)
print('Saved summary CSV. shape:', summary.shape)
print(
    summary[['setting','regime','detector','f1_mean','real_det_mean','synth_det_mean','auroc_mean','normal_fpr_mean']]
    .to_string(index=False)
)


Saved summary CSV. shape: (24, 22)
           setting         regime           detector  f1_mean  real_det_mean  synth_det_mean  auroc_mean  normal_fpr_mean
balanced_detection synthetic_only InceptionTime-lite 0.669635       0.548485        0.900000    0.674827         0.599595
balanced_detection synthetic_only             MOMENT 0.666667       1.000000        1.000000    0.461234         1.000000
balanced_detection synthetic_only             Mantis 0.795995       0.203030        0.921547    0.860960         0.103239
balanced_detection synthetic_only           PatchTST 0.714372       0.201515        0.903315    0.777823         0.288259
balanced_detection synthetic_only      TimesNet-lite 0.654869       0.622727        0.841436    0.661350         0.606883
balanced_detection synthetic_only               Toto 0.755737       0.234848        0.900552    0.831128         0.189069
    controlled_500    all_origins InceptionTime-lite 0.674992       0.928000        0.900000    0.650280       

## 6. LaTeX rows for the paper

We render only the *body rows* of the new detectors so they can be appended
to ``tab:e9_transfer`` (the existing E9 table) without rewriting the header
or the LaTeX preamble.

In [7]:
DETECTOR_PRETTY = {
    'MOMENT':              'MOMENT~\\cite{goswami2024moment} (frozen)',
    'Toto':                'Toto~\\cite{toto2024} (frozen)',
    'Mantis':              'Mantis~\\cite{feofanov2025mantis} (frozen)',
    'TimesNet-lite':       'TimesNet~\\cite{wu2023timesnet} (e2e)',
    'InceptionTime-lite':  'InceptionTime~\\cite{ismail2020inceptiontime} (e2e)',
    'PatchTST':            'PatchTST~\\cite{nie2023patchtst} (e2e)',
}

COLUMN_BLOCKS = [
    ('controlled_500',     'all_origins',    'Ctrl-500 / all'),
    ('controlled_500',     'synthetic_only', 'Ctrl-500 / synth'),
    ('balanced_detection', 'synthetic_only', 'Bal-494 / synth'),
    ('fullscale',          'synthetic_only', 'Full-6.4k / synth'),
]
METRICS = [
    ('f1', 'F1', 'f1'),
    ('auroc', 'AUC', 'f1'),
    ('real_det', 'Real', 'pct'),
    ('synth_det', 'Synth', 'pct'),
]

# Reuse the same renderer to make the full standalone table (helpful for a
# quick LaTeX preview); we will hand-paste only the body rows into the paper.
caption_full = (
    'Supervised SOTA detectors on TelecomTS using the same regime grid as '
    'Table~\\ref{tab:e9_transfer}. Each cell reports mean$\\pm$std over '
    '10 seeds. Foundation backbones (MOMENT, Toto, Mantis) are frozen with a '
    'binary head trained on the regime-filtered training pool; '
    'TimesNet, InceptionTime, and PatchTST are trained end-to-end with '
    'cross-entropy on $\\{normal, anomaly\\}$ labels.'
)
table_tex = sh.render_wide_table(
    summary,
    detector_order=['MOMENT', 'Toto', 'Mantis', 'TimesNet-lite', 'InceptionTime-lite', 'PatchTST'],
    column_blocks=COLUMN_BLOCKS,
    metrics=METRICS,
    label='tab:e14_sup',
    caption=caption_full,
    detector_pretty=DETECTOR_PRETTY,
)
out = TABLES / 'E14_sup_table.tex'
out.write_text(table_tex)
print('Wrote', out)
print('\n----- LaTeX preview (full standalone) -----')
print(table_tex[:2400])


Wrote experiments/E14_supervised_sota_transfer/tables/E14_sup_table.tex

----- LaTeX preview (full standalone) -----
\begin{table*}[t]
  \caption{Supervised SOTA detectors on TelecomTS using the same regime grid as Table~\ref{tab:e9_transfer}. Each cell reports mean$\pm$std over 10 seeds. Foundation backbones (MOMENT, Toto, Mantis) are frozen with a binary head trained on the regime-filtered training pool; TimesNet, InceptionTime, and PatchTST are trained end-to-end with cross-entropy on $\{normal, anomaly\}$ labels.}
  \label{tab:e14_sup}
  \footnotesize
  \setlength{\tabcolsep}{1.5pt}
  \begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}lcccccccccccccccc@{}}
    \toprule
    Detector & \multicolumn{4}{c}{Ctrl-500 / all} & \multicolumn{4}{c}{Ctrl-500 / synth} & \multicolumn{4}{c}{Bal-494 / synth} & \multicolumn{4}{c}{Full-6.4k / synth} \\
    \cmidrule(lr){2-5}\cmidrule(lr){6-9}\cmidrule(lr){10-13}\cmidrule(lr){14-17}
      & F1 & AUC & Real & Synth & F1 & AUC & Real & Synth & F1 & AU

In [8]:
# Body-only rendering for direct paste into tab:e9_transfer.
def _fmt(mean, std, kind):
    return sh._fmt_pair(mean, std, kind)


summary_indexed = summary.set_index(['setting', 'regime', 'detector'])
body_rows = []
for det in ['MOMENT', 'Toto', 'Mantis', 'TimesNet-lite', 'InceptionTime-lite', 'PatchTST']:
    cells = [DETECTOR_PRETTY.get(det, det)]
    for setting, regime, _ in COLUMN_BLOCKS:
        for col, _, kind in METRICS:
            try:
                row = summary_indexed.loc[(setting, regime, det)]
                mean = row[f'{col}_mean']; std = row[f'{col}_std']
            except KeyError:
                mean = float('nan'); std = float('nan')
            cells.append(_fmt(mean, std, kind))
    body_rows.append('    ' + ' & '.join(cells) + ' \\\\')

body_tex = '\n'.join(body_rows) + '\n'
(TABLES / 'E14_sup_body_rows.tex').write_text(body_tex)
print('Wrote', TABLES / 'E14_sup_body_rows.tex')
print('\n----- body rows (paste under \\midrule of tab:e9_transfer) -----')
print(body_tex)


Wrote experiments/E14_supervised_sota_transfer/tables/E14_sup_body_rows.tex

----- body rows (paste under \midrule of tab:e9_transfer) -----
    MOMENT~\cite{goswami2024moment} (frozen) & 0.67{\tiny$\pm$0.00} & 0.44{\tiny$\pm$0.09} & 100{\tiny$\pm$0}\% & 99{\tiny$\pm$3}\% & 0.66{\tiny$\pm$0.01} & 0.44{\tiny$\pm$0.09} & 98{\tiny$\pm$8}\% & 98{\tiny$\pm$5}\% & 0.67{\tiny$\pm$0.00} & 0.46{\tiny$\pm$0.08} & 100{\tiny$\pm$0}\% & 100{\tiny$\pm$0}\% & 0.06{\tiny$\pm$0.03} & 0.49{\tiny$\pm$0.08} & 80{\tiny$\pm$42}\% & 79{\tiny$\pm$42}\% \\
    Toto~\cite{toto2024} (frozen) & 0.74{\tiny$\pm$0.07} & 0.82{\tiny$\pm$0.07} & 83{\tiny$\pm$13}\% & 78{\tiny$\pm$12}\% & 0.53{\tiny$\pm$0.06} & 0.68{\tiny$\pm$0.04} & 18{\tiny$\pm$7}\% & 72{\tiny$\pm$13}\% & 0.76{\tiny$\pm$0.01} & 0.83{\tiny$\pm$0.00} & 23{\tiny$\pm$5}\% & 90{\tiny$\pm$3}\% & 0.59{\tiny$\pm$0.01} & 0.88{\tiny$\pm$0.00} & 0{\tiny$\pm$0}\% & 68{\tiny$\pm$4}\% \\
    Mantis~\cite{feofanov2025mantis} (frozen) & 0.76{\tiny$\pm$0.04} & 0.82{\ti

## 7. Inline Markdown summary

In [9]:
def fmt_pct(mean, std):
    if pd.isna(mean): return '--'
    if pd.isna(std) or std < 1e-9: return f'{100*mean:.1f}'
    return f'{100*mean:.1f}±{100*std:.1f}'

def fmt_f1(mean, std):
    if pd.isna(mean): return '--'
    if pd.isna(std) or std < 1e-9: return f'{mean:.3f}'
    return f'{mean:.3f}±{std:.3f}'

print(f"{'Detector':>20s} | " + ' | '.join([f'{lbl:>26s}' for _, _, lbl in COLUMN_BLOCKS]))
for det in ['MOMENT', 'Toto', 'Mantis', 'TimesNet-lite', 'InceptionTime-lite', 'PatchTST']:
    cells = []
    for setting, regime, _ in COLUMN_BLOCKS:
        try:
            row = summary_indexed.loc[(setting, regime, det)]
            cells.append(
                f"F1 {fmt_f1(row['f1_mean'], row['f1_std'])} "
                f"R {fmt_pct(row['real_det_mean'], row['real_det_std'])} "
                f"S {fmt_pct(row['synth_det_mean'], row['synth_det_std'])}"
            )
        except KeyError:
            cells.append('--')
    print(f"{det:>20s} | " + ' | '.join(c.ljust(26) for c in cells))


            Detector |             Ctrl-500 / all |           Ctrl-500 / synth |            Bal-494 / synth |          Full-6.4k / synth
              MOMENT | F1 0.667 R 100.0 S 99.2±2.5 | F1 0.662±0.013 R 97.6±7.6 S 98.4±5.1 | F1 0.667 R 100.0 S 100.0   | F1 0.059±0.031 R 79.6±42.0 S 79.2±41.8
                Toto | F1 0.743±0.066 R 82.8±12.5 S 78.4±11.8 | F1 0.534±0.057 R 17.6±7.1 S 72.0±12.6 | F1 0.756±0.010 R 23.5±4.5 S 90.1±3.3 | F1 0.588±0.014 R 0.0 S 67.8±4.1
              Mantis | F1 0.756±0.044 R 90.0±5.4 S 71.6±14.7 | F1 0.548±0.114 R 26.8±16.9 S 63.6±12.9 | F1 0.796±0.011 R 20.3±4.2 S 92.2±2.5 | F1 0.681±0.028 R 5.4±2.7 S 74.8±6.4
       TimesNet-lite | F1 0.648±0.022 R 92.4±12.4 S 91.6±11.8 | F1 0.581±0.105 R 67.2±24.1 S 73.6±23.1 | F1 0.655±0.011 R 62.3±7.4 S 84.1±3.9 | F1 0.362±0.035 R 0.0 S 38.5±7.1
  InceptionTime-lite | F1 0.675±0.018 R 92.8±9.8 S 90.0±9.8 | F1 0.560±0.087 R 39.6±26.5 S 80.0±17.9 | F1 0.670±0.022 R 54.8±19.1 S 90.0±7.3 | F1 0.358±0.129 R 0.7±1.2 S 39.